# Huấn luyện Faster R-CNN cho nhận diện biển báo (Zalo AI 2020)
Notebook này được thiết kế để chạy trên **Kaggle** (GPU P100/T4). Nó sử dụng PyTorch thuần để xây dựng DataLoader và vòng lặp huấn luyện.

In [ ]:
import os
import glob

# Tự động tìm đường dẫn dataset trên Kaggle
print("Đang tìm kiếm dataset trên Kaggle...")
json_paths = glob.glob('/kaggle/input/**/train_traffic_sign_dataset.json', recursive=True)
if not json_paths:
    raise FileNotFoundError("Không tìm thấy file JSON. Vui lòng Add Dataset za-traffic-2020 vào Kaggle!")

json_file = json_paths[0]
root_dir = os.path.dirname(json_file).replace('traffic_train', 'traffic_train/images')

if not os.path.exists(root_dir):
    img_dirs = glob.glob('/kaggle/input/**/traffic_train/images', recursive=True)
    if img_dirs:
        root_dir = img_dirs[0]

print(f"File JSON: {json_file}\n")
print(f"Thư mục ảnh: {root_dir}\n")


In [ ]:
# Tải thư viện cần thiết
import os
import json
import torch
import torch.utils.data
from PIL import Image
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F

# Kiểm tra xem GPU có sẵn sàng không
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Đang sử dụng thiết bị: {device}")
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.cluster import KMeans
import numpy as np


In [ ]:
# Khoi tao chuoi Augmentation
def get_transform():
    return A.Compose([
        # Cắt khuôn 512x512 (Power of 2) phù hợp ảnh gốc 626px
        A.RandomSizedBBoxSafeCrop(width=512, height=512, erosion_rate=0.0, p=0.3),
        A.Normalize(),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'], min_visibility=0.5))



In [ ]:
import cv2
import torch
from torch.utils.data import Dataset

class ZaloTrafficDataset(Dataset):
    """
    Lớp Dataset đọc dữ liệu trực tiếp từ file JSON chuẩn COCO của Zalo AI.
    Chuyển đổi Bounding Box từ hệ tọa độ COCO [x, y, w, h] sang Pascal VOC [xmin, ymin, xmax, ymax]
    để tương thích với chuẩn đầu vào của Faster R-CNN.
    """
    def __init__(self, root_dir, json_file, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        with open(json_file, 'r', encoding='utf-8') as f:
            self.coco_data = json.load(f)
            
        # Gom nhóm annotations theo image_id để truy xuất siêu tốc O(1)
        self.image_annotations = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.image_annotations:
                self.image_annotations[img_id] = []
            self.image_annotations[img_id].append(ann)
            
        self.images = self.coco_data['images']
        
    def __len__(self):
        return len(self.images)
        
    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']
        img_name = img_info['file_name']
        img_path = os.path.join(self.root_dir, img_name)
        
        # Đọc ảnh bằng OpenCV và chuyển hệ màu BGR -> RGB
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        anns = self.image_annotations.get(img_id, [])
        
        boxes = []
        labels = []
        
        for ann in anns:
            # COCO bbox: [x, y, width, height]
            x, y, w, h = ann['bbox']
            # Ép sang Pascal VOC: [xmin, ymin, xmax, ymax]
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id']) # 1 đến 7
            
        # Xử lý an toàn nếu ảnh không có biển báo nào
        if len(boxes) == 0:
            boxes = np.zeros((0, 4), dtype=np.float32)
            labels = np.zeros((0,), dtype=np.int64)
            
        # Áp dụng Augmentation (BBox-Safe Crop)
        if self.transform:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image = transformed['image'] # Trở thành Tensor (C, H, W)
            boxes = transformed['bboxes']
            labels = transformed['labels']
            
        # Đóng gói dữ liệu thành Tensor PyTorch
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([img_id])
        
        # Khắc phục lỗi tensor rỗng sau khi crop
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            
        # Chuẩn hóa cấu trúc Target cho Faster R-CNN
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id
        }
        
        return image, target


In [ ]:
from torch.utils.data import random_split

def collate_fn(batch):
    return tuple(zip(*batch))

full_dataset = ZaloTrafficDataset(root_dir, json_file, transform=get_transform())

# Chia dữ liệu 90% Train, 10% Validation
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn
)
print(f"Đã chia dữ liệu: {len(train_dataset)} Train | {len(val_dataset)} Validation\n")

# [TỪ EDA E6] TỰ ĐỘNG CHẠY K-MEANS ĐỂ TÌM ANCHOR BOX
print("Đang phân tích K-Means 5 cụm cho Anchor Box từ tập dữ liệu...")
all_boxes = []
for ann in full_dataset.coco_data['annotations']:
    w, h = ann['bbox'][2], ann['bbox'][3]
    all_boxes.append([w, h])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(all_boxes)
centers = np.sort(kmeans.cluster_centers_, axis=0)
anchor_sizes_kmeans = tuple(int(center[0]) for center in centers)
print(f"5 Kích thước Anchor thu được: {anchor_sizes_kmeans}\n")

# FPN cần 5 tuple riêng biệt cho 5 level
ANCHOR_SIZES = tuple((size,) for size in anchor_sizes_kmeans)
ASPECT_RATIOS = ((1.0,),) * len(ANCHOR_SIZES) # Tỷ lệ 1:1 cho tất cả 5 level


In [ ]:
from torchvision.models.detection.anchor_utils import AnchorGenerator

# Hàm tạo mô hình Faster R-CNN với K-Means Anchors
def get_model(num_classes, anchor_sizes, aspect_ratios):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    
    # Ép K-Means Anchor Generator vào RPN
    anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=aspect_ratios)
    model.rpn.anchor_generator = anchor_generator
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model



In [ ]:
# Gọi thư viện hiển thị thanh tiến trình
from tqdm import tqdm
import os

# Cấu hình lưu trữ trên Kaggle
save_dir = '/kaggle/working/faster_rcnn_highres'
os.makedirs(save_dir, exist_ok=True) 

# Chuẩn bị huấn luyện
num_classes = 8  # [QUAN TRỌNG] 7 class biển báo + 1 class nền (background)
model = get_model(num_classes, ANCHOR_SIZES, ASPECT_RATIOS)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
# Giữ nguyên SGD thay vì AdamW để đảm bảo ResNet hội tụ tốt nhất
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Thêm bộ hạ tốc StepLR: Đến Epoch thứ 10 thì chia LR cho 10
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

num_epochs = 15 
print("Bắt đầu huấn luyện mô hình Faster R-CNN...")

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ==========================
    # PHA 1: TRAINING
    # ==========================
    model.train()  
    train_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    
    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        train_loss += losses.item()
        progress_bar.set_postfix(loss=losses.item())

    avg_train_loss = train_loss / len(train_loader)
    
    # ==========================
    # PHA 2: VALIDATION (Mẹo no_grad với FrozenBatchNorm)
    # ==========================
    val_loss = 0
    # Giữ nguyên model.train() để ép trả về hàm Loss
    # Dùng no_grad() để chặn gradient (không học lỏm dữ liệu val)
    with torch.no_grad():
        for images, targets in val_loader:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            val_loss += losses.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"\nEpoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # ==========================
    # PHA 3: LƯU BEST MODEL & STEP LR
    # ==========================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_path = os.path.join(save_dir, 'faster_rcnn_best.pth')
        torch.save(model.state_dict(), best_path)
        print(f">> Đã lưu kỷ lục Best Model mới! (Val Loss: {best_val_loss:.4f})")
        
    last_path = os.path.join(save_dir, 'faster_rcnn_last.pth')
    torch.save(model.state_dict(), last_path)
    
    # Cập nhật Learning Rate
    lr_scheduler.step()

print(f"\nHoàn tất huấn luyện. Mô hình tốt nhất (Best) đã được bảo lưu tại {save_dir}/faster_rcnn_best.pth")


In [ ]:
# TỰ ĐỘNG NÉN VÀ ÉP TRÌNH DUYỆT TẢI VỀ (AUTO-DOWNLOAD)
from IPython.display import HTML

print("Đang nén kết quả. Vui lòng đợi...")
!zip -r -q /kaggle/working/faster_rcnn_results.zip /kaggle/working/faster_rcnn_highres
print("Nén xong! Trình duyệt sẽ tự động tải file về ngay bây giờ...")

# Dùng JavaScript tạo một thẻ <a> ẩn và tự động click vào nó
html_code = """
<a id="auto_download" href="faster_rcnn_results.zip" download>Đang tải xuống...</a>
<script>
    document.getElementById("auto_download").click();
</script>
"""
HTML(html_code)